# Workflow 18: verify checkpoints and complete accuracy intervals
This notebook reads corrected inference results and can rerun registered checkpoints on the frozen historical test partition. It does not train models.

The original CSVs confirm ResNet50 97.49%, EfficientNetB0 uniform 89.09% and EfficientNetB2 uniform 93.95%. Restoring nested Normalization state after legacy H5 loading resolves the EfficientNet reload differences and recovers all original class predictions. ResNet50 reload still differs on one image (97.35%); keep its original CSV-based result separate. The 90.56% classical refit does not replace the 90.71% original spreadsheet result.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
import pandas as pd
from IPython.display import display
assert sys.platform.startswith("linux"), "Select the BRISC Linux (GPU) kernel."
ROOT = Path(os.environ.get("BRISC_REPO", "/mnt/c/Users/cbot/Desktop/brain_tumor_brisc_tmp"))
DATASET = Path(os.environ.get("BRISC_DATASET", "/root/brisc/data/brisc2025_clean/classification_task"))
PYTHON = sys.executable
print("Repository:", ROOT, "\nPython:", PYTHON)
subprocess.run([PYTHON, "-c", "import tensorflow as tf; g=tf.config.list_physical_devices('GPU'); print(g); assert g, 'GPU missing'"], check=True)


## Corrected inference results already available


In [ ]:
BASE = ROOT / "results/extensions_2026-10-07/inference"
latest = pd.DataFrame(json.loads((BASE / "metrics.json").read_text())).T.rename_axis("model").reset_index()
expected = {"convnext":94.99,"resnet50":97.49,"efficientnetb2_uniform":93.95,"efficientnetb0_uniform":89.09,"cnn_four_block":91.89,"efficientnetb0_optimized":98.53,"efficientnetb1_optimized":98.82,"efficientnetb2_optimized":98.23}
latest["reported_accuracy"] = latest.model.map(expected)
latest["matches_reported_rounding"] = latest.accuracy.astype(float).round(2).eq(latest.reported_accuracy)
display(latest[["model","accuracy","reported_accuracy","matches_reported_rounding","ci95_lo","ci95_hi"]])
print("EfficientNet H5 normalization-state loading has been corrected. Original ResNet50 CSV confirms 97.49%; reload differs on one image.")

## Optional: rerun inference
Run only when you want a fresh verification. The output directory must be new. Batch size 8 avoids the memory failures encountered with batch size 32. Run this notebook's GPU cell separately from Workflow 17's training cell.


In [ ]:
from datetime import datetime
OUT = ROOT / "runs" / ("verified_inference_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
models = ["convnext", "resnet50", "efficientnetb2_uniform", "efficientnetb0_uniform", "cnn_four_block",
          "efficientnetb0_optimized", "efficientnetb1_optimized", "efficientnetb2_optimized"]
subprocess.run([PYTHON, "-u", str(ROOT / "scripts/evaluate_models.py"),
    "--dataset", str(DATASET), "--models", *models, "--batch-size", "8", "--output", str(OUT)], check=True, cwd=ROOT)
display(pd.DataFrame(json.loads((OUT / "metrics.json").read_text())).T)
